In [1]:
import numpy as np
from typing import Dict
from utils import utils
from utils.data_utils import load_metadata_and_map

In [2]:
METADATA_FILE = "embeddings/id_metadata.json"
TENSOR_DIR = "embeddings/uuid_embeddings/embeddings/"

In [3]:
# 1. Load Tensors
tensors = utils.load_tensor(TENSOR_DIR, num_workers=16)


Loading cache from embeddings/cache


In [4]:
# 2. Load Metadata (Shared Logic)
df, artist_to_indices = load_metadata_and_map(METADATA_FILE, tensors)

Loading metadata from embeddings/id_metadata.json...
Building Artist to Track Index map...


In [5]:
# 3. Pool Embeddings
print("Pooling embeddings (mean+max)...")
all_embeddings = utils.pool_loaded_tensor_dict(tensors=tensors, mode='mean+max')


Pooling embeddings (mean+max)...
Detected allchunks tensors, pooling to 1D embedding (mean+max)


100%|██████████| 138748/138748 [00:04<00:00, 33671.55it/s]


In [6]:
# Convert to standard numpy matrix
embedding_mat = np.array(list(all_embeddings.values()))

In [7]:
# 4. Compute Centroids
print("Computing Artist Centroids...")
artist_centroids: Dict[str, np.ndarray] = {}

for artist_id, indices in artist_to_indices.items():
  # Select specific tracks for this artist
  track_embeds = embedding_mat[np.array(indices)]
  # Compute mean
  artist_centroids[artist_id] = track_embeds.mean(axis=0)


Computing Artist Centroids...


In [8]:
from utils.data_utils import knn_search, get_artist_name
# 5. Search
QUERY_ARTIST_ID = '85de2a90-ea44-473f-af2f-2c22dcc99064'

print(f"Searching 10 nearest neighbors for {QUERY_ARTIST_ID} (Centroid)...")
if QUERY_ARTIST_ID not in artist_centroids:
  print(f"Error: Artist ID {QUERY_ARTIST_ID} not found.")

result_ids = knn_search(
  artist_centroids, 
  artist_centroids[QUERY_ARTIST_ID], 
  k=10, 
  backend=np
)

for res_id in result_ids:
  print(f"- {get_artist_name(df, res_id)}")


Searching 10 nearest neighbors for 85de2a90-ea44-473f-af2f-2c22dcc99064 (Centroid)...
- TAMUSIC
- 外柿山
- 吉田未来Project
- Colorful Harmony
- Whisper Records
- Ridil
- BITPLANE
- まいしんドリラー
- 豆屋
- Secret Messenger


In [9]:
import pandas as pd

val_data = pd.read_json("data/validation/top_1000_artists_by_variance.json")
val_data

,Id,Name
0,00a577e0-ccd9-4463-8fc6-f77ae6e99d53,Pure Highball
1,af62098a-1ffd-499b-bc2d-4d1b2bcc33ad,アールグレイ
2,fafe73a1-168a-4855-9e51-e070182a1bd2,Re：Volte
3,3e72f0ef-c795-4573-9265-13e3c1298e1c,まさかど☆くらいしすっ！
4,a9d6f2a9-55cd-4286-a041-60993b1e3eea,2nd Flush
...,...,...
609,faa689b4-b345-4e0d-be19-066eebf55a56,岸田教団&THE明星ロケッツ
610,edc476dd-b993-47ce-bec6-2aae1dce30d3,FalKKonE
611,b08bb9b5-4e2e-45e2-9e39-eb65bea3ee49,鉄腕トカゲ探知機
612,c2e3595f-672e-4cb8-a7ff-e984735370d8,A-GEAR


In [10]:
results = []

import tqdm

print("Generating results for validation set...")
for _, row in tqdm.tqdm(val_data.iterrows()):
  artist_id = row["Id"]
  artist_name = row["Name"]
  if artist_id not in artist_centroids:
    print(f"Error: Artist ID {artist_id} not found.")
    continue
  result_ids = knn_search(
    artist_centroids, 
    artist_centroids[artist_id], 
    k=10, 
    backend=np
  )

  results.append({
    "ArtistID": artist_id,
    "Name": artist_name,
    "Results": result_ids
  })

results = pd.DataFrame(results)

results.head()

Generating results for validation set...


614it [00:04, 143.26it/s]


,ArtistID,Name,Results
0,00a577e0-ccd9-4463-8fc6-f77ae6e99d53,Pure Highball,"[00a577e0-ccd9-4463-8fc6-f77ae6e99d53, 4a426d1..."
1,af62098a-1ffd-499b-bc2d-4d1b2bcc33ad,アールグレイ,"[af62098a-1ffd-499b-bc2d-4d1b2bcc33ad, 59bc148..."
2,fafe73a1-168a-4855-9e51-e070182a1bd2,Re：Volte,"[fafe73a1-168a-4855-9e51-e070182a1bd2, 4b5bd52..."
3,3e72f0ef-c795-4573-9265-13e3c1298e1c,まさかど☆くらいしすっ！,"[3e72f0ef-c795-4573-9265-13e3c1298e1c, 33c5902..."
4,a9d6f2a9-55cd-4286-a041-60993b1e3eea,2nd Flush,"[a9d6f2a9-55cd-4286-a041-60993b1e3eea, f730287..."


In [12]:
# save results
results.to_json("data/validation/centroid_results.jsonl", orient="records", lines=True)
